In [ ]:
# OPTIONAL: Install
# pip install -qU langchain langchain-openai python-dotenv

## Tutorial: Adding Cache to a LangChain Project with InMemoryCache
Speed up repeated queries with a simple, drop‑in cache.


In [ ]:
from getpass import getpass
from langchain_openai import ChatOpenAI

# Enter your OpenRouter API key securely when prompted.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

# OpenRouter provides an OpenAI-compatible API.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# You can change this to any compatible OpenRouter model.
MODEL = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    seed=42,
)

print("OpenRouter configured successfully.")
print("Model:", MODEL)
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, seed=42)

qa_prompt = PromptTemplate.from_template(
    "Answer briefly: {question}"
)
qa_chain = LLMChain(llm=llm, prompt=qa_prompt)


### Step 1: Run the chain twice without cache
We’ll measure elapsed time and compare after enabling cache.


In [ ]:
q = "What is LangChain in one sentence?"

start = time.time()
print(qa_chain.run({"question": q}))
print(f"First run: {time.time() - start:.2f}s")

start = time.time()
print(qa_chain.run({"question": q}))
print(f"Second run (no cache): {time.time() - start:.2f}s")


### Step 2: Enable InMemoryCache
We’ll set a global LLM cache and repeat the calls to see the speedup.


In [ ]:
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache

set_llm_cache(InMemoryCache())

start = time.time()
print(qa_chain.run({"question": q}))
print(f"Third run (cached): {time.time() - start:.2f}s")

start = time.time()
print(qa_chain.run({"question": q}))
print(f"Fourth run (cached): {time.time() - start:.2f}s")


### Step 3: Notes on production caches
- Use Redis/SQLite/Postgres caches for persistence across processes and restarts
- Consider selective caching (e.g., only cache deterministic prompts)
- Invalidate on data or prompt changes to avoid stale answers